# Cross-dataset DPE study, step 2: strictly nested post-hoc procedures (CPU)

Plan: `experiments/dpe_cross_dataset/PLAN.md`. Select **CPU**, **Internet off** is fine.
**Add Input -> Notebook Output** for every backbone session of `DATASET` (the outputs of
`backbones_kaggle.ipynb`). To continue an interrupted run, also attach this notebook's own earlier output.

For runs that need several sessions, set **Settings -> Environment -> Pin to original environment**:
the run binds its checkpoints to the Python/NumPy/SciPy/torch versions and refuses to mix them.

`MODE`:
* `'smoke'`: 3-head ensembles, 2 epochs, split 0, one backbone. Checks the pipeline; never scores the test set.
* `'timing'`: full recipe, split 0, one backbone, test **not** scored (PLAN step 0.4). Its checkpoints are reused.
* `'run'`: the declared run (all backbones, splits 0-9). The summary is printed only when every cell is done
  (PLAN Sect. 8.5); otherwise save the version, attach its output and run again to continue.

The code is the dataset-agnostic port of the Waterbirds strict nested run (`xdpe`), checked bit for bit
against the original on Waterbirds features (`code/tests/equivalence.py`).

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, shutil, glob
DATASET = 'CelebA'          # 'CelebA' | 'CivilCommentsFine' | 'MultiNLI'
MODE = 'smoke'              # 'smoke' | 'timing' | 'run'
VAL_SUBSAMPLE = 0           # 1199 for the declared secondary validation-size analysis (CelebA, PLAN Sect. 7)
WORKERS = os.cpu_count() or 4
TIME_LIMIT_HOURS = 11.0
WORK = Path('/kaggle/working/xdpe-nested')
CODE = WORK / 'code'
(CODE / 'xdpe').mkdir(parents=True, exist_ok=True)
tag = DATASET + (f'-val{VAL_SUBSAMPLE}' if VAL_SUBSAMPLE else '')
OUT = WORK / ('smoke-' + tag if MODE == 'smoke' else tag)
import numpy, scipy, torch
print('python', sys.version.split()[0], 'numpy', numpy.__version__, 'scipy', scipy.__version__, 'torch', torch.__version__, 'cpus', WORKERS)

## Code (`xdpe` package and the declared analysis)

In [ ]:
%%writefile "{CODE}/xdpe/__init__.py"
"""Dataset-agnostic port of the strictly nested DPE post-hoc study (protocol strict-nested-postmean-v2).

Every function here mirrors one in experiments/dpe_nested_corrected/code (corrected_nested.py,
nested_selection.py) or src/methods/dpe_independent / dpe_head_reweighting, with the four
Waterbirds groups and two classes replaced by the groups and classes found in the data.
For Waterbirds inputs the arithmetic is unchanged; tests/test_equivalence.py checks this
bit for bit against the original code.
"""

In [ ]:
%%writefile "{CODE}/xdpe/calibrate.py"
"""Binary calibrators, metrics and the validation-only selection rules.

Ported from experiments/dpe_nested_corrected/code/{nested_selection,corrected_nested}.py and
src/methods/dpe_independent/fitting.py (ResidualModel, fit_adam, predict_adam). Group means
run over the groups present, in sorted order (the Waterbirds order for Waterbirds data).
"""
import contextlib

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

SETTINGS = [("average", 0.1), ("maximum", 0.1), ("average", 0.01), ("maximum", 0.01)]
FIXED = ("average", 0.1)
RULES = ("nll", "wga")


def name(setting):
    return f"{setting[0]}-l{setting[1]:g}"


def names_of(groups):
    return tuple(sorted(set(np.asarray(groups).tolist())))


# ----------------------------------------------------------------------------- metrics

def group_accuracy(pred, y, groups):
    groups = np.asarray(groups)
    correct = np.asarray(pred) == np.asarray(y)
    return {g: float(correct[groups == g].mean()) for g in names_of(groups)}


def record(pred, y, groups, **extra):
    acc = group_accuracy(pred, y, groups)
    groups = np.asarray(groups)
    out = dict(wga=min(acc.values()) * 100, overall=float((np.asarray(pred) == y).mean()) * 100,
               groups={g: v * 100 for g, v in acc.items()},
               counts={g: int((groups == g).sum()) for g in acc})
    out.update(extra)
    return out


def balanced_nll(prob1, y, groups):
    p = np.clip(np.asarray(prob1, dtype=np.float64), 1e-12, 1 - 1e-12)
    loss = -(y * np.log(p) + (1 - y) * np.log1p(-p))
    return float(np.mean([loss[groups == g].mean() for g in names_of(groups)]))


def inner_score(prob1, y, groups):
    acc = group_accuracy(prob1 > .5, y, groups)
    return dict(wga=min(acc.values()) * 100, nll=balanced_nll(prob1, y, groups))


def choose(scores, rule):
    """scores: {setting_name: {wga, nll}}. Deterministic, validation-only."""
    order = [name(s) for s in SETTINGS]
    if rule == "nll":
        key = lambda n: (round(scores[n]["nll"], 9), -round(scores[n]["wga"], 6), order.index(n))
    elif rule == "wga":
        key = lambda n: (-round(scores[n]["wga"], 6), round(scores[n]["nll"], 9), order.index(n))
    else:
        raise ValueError(rule)
    return min((n for n in order if n in scores), key=key)


# ----------------------------------------------------------------------------- calibrators

def logit_clip(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)


def fit_bias(x, y, g, setting):
    """Offset-only recalibration sigmoid(logit(p) + b), group-balanced loss, no scale."""
    from scipy.optimize import minimize_scalar
    z, yy = logit_clip(x[:, 0]), np.asarray(y, dtype=np.float64)
    loss_name, penalty = setting
    names = names_of(g)

    def objective(b):
        margin = (2 * yy - 1) * (z + b)
        per = np.logaddexp(0.0, -margin)
        groups = [per[g == k].mean() for k in names]
        return (np.mean(groups) if loss_name == "average" else np.max(groups)) + penalty * b * b

    result = minimize_scalar(objective, bounds=(-5.0, 5.0), method="bounded", options=dict(xatol=1e-6))
    return dict(kind="bias", b=float(result.x))


@contextlib.contextmanager
def cpu_randomness(seed):
    threads = torch.get_num_threads()
    try:
        torch.set_num_threads(1)
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(seed)
            yield
    finally:
        torch.set_num_threads(threads)


class ResidualModel(nn.Module):
    def __init__(self, kind, heads=15, hidden=8):
        super().__init__()
        if kind not in ("affine", "linear"):
            raise ValueError("Unknown residual model")
        self.kind, self.heads, self.hidden = kind, heads, hidden
        self.a = nn.Parameter(torch.ones((), dtype=torch.float64))
        self.b = nn.Parameter(torch.zeros((), dtype=torch.float64))
        if kind == "linear":
            self.correction = nn.Linear(heads, 1, bias=False).double()
            nn.init.zeros_(self.correction.weight)

    def logits(self, q):
        mean = q.mean(1).clamp(1e-12, 1 - 1e-12)
        result = self.a * torch.logit(mean) + self.b
        if hasattr(self, "correction"):
            disagreement = q - q.mean(1, keepdim=True)
            result = result + (self.correction(disagreement) - self.correction(torch.zeros_like(disagreement))).squeeze(1)
        return result

    def regularizer(self):
        result = (self.a - 1).square() + self.b.square()
        if hasattr(self, "correction"):
            result = result + torch.cat([p.reshape(-1) for p in self.correction.parameters()]).square().mean()
        return result


def group_losses(values, index, count):
    return torch.stack([values[index == i].mean() for i in range(count)])


def fit_adam(inputs, y, groups, kind, setting, cfg, seed):
    """Fixed 400-step Adam fit of the affine or linear-residual map."""
    y, groups = np.asarray(y), np.asarray(groups)
    names = names_of(groups)
    x = torch.tensor(np.asarray(inputs), dtype=torch.float64)
    target = torch.tensor(y, dtype=torch.float64)
    g = torch.tensor([names.index(value) for value in groups])
    if not torch.isfinite(x).all():
        raise ValueError("Invalid fitting inputs")
    loss_name, penalty = setting
    with cpu_randomness(seed):
        model = ResidualModel(kind, x.shape[1], cfg["hidden_units"])
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
        for step in range(cfg["steps"] + 1):
            logits = model.logits(x)
            per_example = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
            losses = group_losses(per_example, g, len(names))
            data_loss = losses.mean() if loss_name == "average" else losses.max()
            objective = data_loss + penalty * model.regularizer()
            if not torch.isfinite(objective) or any(not torch.isfinite(p).all() for p in model.parameters()):
                raise ValueError("Nonfinite fit; no candidate accepted")
            if step == cfg["steps"]:
                break
            optimizer.zero_grad(set_to_none=True)
            objective.backward()
            optimizer.step()
        state = {k: v.detach().tolist() for k, v in model.state_dict().items()}
    return dict(kind=kind, state=state, heads=x.shape[1], hidden=cfg["hidden_units"], seed=seed, steps=cfg["steps"])


def predict_adam(model, inputs, cfg):
    with cpu_randomness(0):
        m = ResidualModel(model["kind"], model["heads"], model["hidden"])
        m.load_state_dict({k: torch.tensor(v, dtype=torch.float64) for k, v in model["state"].items()}, strict=True)
        with torch.inference_mode():
            q = m.logits(torch.tensor(np.asarray(inputs), dtype=torch.float64)).sigmoid().numpy()
    # The original returned probabilities([1-q, q])[:, 1] after validation; that is q.
    if not np.isfinite(q).all():
        raise ValueError("Nonfinite calibrated probabilities")
    return q


def fit_model(x, y, g, kind, setting, cfg, seed):
    if kind == "bias":
        return fit_bias(x, y, g, setting)
    return fit_adam(x, y, g, kind, setting, cfg, seed)


def predict_model(model, x, cfg):
    if model.get("kind") == "bias":
        return 1.0 / (1.0 + np.exp(-(logit_clip(x[:, 0]) + model["b"])))
    return predict_adam(model, x, cfg)


# ----------------------------------------------------------------------------- thresholds

def fit_threshold_wga(x, y, g):
    """Legacy coarse logit-offset grid (diagnostic only)."""
    z, yy = logit_clip(x[:, 0]), np.asarray(y)
    best = None
    for b in np.arange(-4.0, 4.0001, 0.05):
        pred = (z + b) > 0
        wga = min((pred[g == k] == yy[g == k]).mean() for k in names_of(g))
        key = (-round(wga, 9), abs(b))
        if best is None or key < best[0]:
            best = (key, float(b))
    return dict(kind="bias", b=best[1])


def fit_threshold_exact(x, y, g):
    """Exact empirical WGA maximiser for p > t over every distinct prediction partition."""
    p = np.asarray(x, dtype=np.float64).reshape(-1)
    y, g = np.asarray(y), np.asarray(g)
    if p.shape != y.shape or y.shape != g.shape or not len(p) or not np.isfinite(p).all():
        raise ValueError("Invalid threshold inputs")
    if (p < 0).any() or (p > 1).any() or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected probabilities and binary labels")
    order = np.argsort(p, kind="stable")
    sorted_p = p[order]
    unique = np.unique(sorted_p)
    candidates = np.unique(np.r_[np.nextafter(unique, -np.inf), unique, .5])
    cut = np.searchsorted(sorted_p, candidates, side="right")
    group_acc = []
    for group in names_of(g):
        mask = g[order] == group
        changes = np.where(mask, 1 - 2 * y[order], 0)
        prefix = np.r_[0, np.cumsum(changes)]
        correct = np.sum(mask & (y[order] == 1)) + prefix[cut]
        group_acc.append(correct / mask.sum())
    wga = np.min(group_acc, axis=0)
    best = min(range(len(candidates)), key=lambda i: (-round(float(wga[i]), 12),
                                                     abs(float(candidates[i]) - .5), float(candidates[i])))
    t = float(candidates[best])
    default = min(group_accuracy(p > .5, y, g).values()) * 100
    coarse = fit_threshold_wga(p[:, None], y, g)
    coarse_wga = min(group_accuracy(logit_clip(p) + coarse["b"] > 0, y, g).values()) * 100
    return dict(kind="exact_threshold", threshold=t, fit_wga=float(wga[best] * 100),
                default_fit_wga=default, legacy_grid_fit_wga=coarse_wga,
                fit_prediction_changes=int(np.sum((p > t) != (p > .5))))


def apply_threshold(model, p):
    return np.asarray(p) > model["threshold"]

In [ ]:
%%writefile "{CODE}/xdpe/data.py"
"""Feature caches, groups, stratified folds and the 60/40 partition.

Ported from src/methods/dpe_independent/sources.py. Groups are "<y><a>" strings found in the
data, iterated in sorted order; for Waterbirds that is ("00", "01", "10", "11") as before.

Duplicated inputs: CivilCommentsFine repeats a comment once per identity it mentions, as
separate rows. Rows that share a `cluster` value are forced into the same fold / partition
side (the side of the cluster's first row in id order), so no comment is used to fit a stage
that then scores a copy of it. Without duplicates (cluster == id) nothing changes.
"""
import hashlib
from pathlib import Path

import numpy as np

FEATURE_KEYS = ("ids", "features", "y", "a", "cluster")


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def group_names(rows):
    return tuple(sorted({r["group"] for r in rows}))


def _keep_clusters_together(rows, assignment):
    """Give every row of a cluster the value of the cluster's first row (sorted by id)."""
    first = {}
    for i in sorted(range(len(rows)), key=lambda i: rows[i]["id"]):
        first.setdefault(rows[i].get("cluster", rows[i]["id"]), i)
    return np.array([assignment[first[r.get("cluster", r["id"])]] for r in rows])


def folds(rows, count=5, seed=0):
    rng = np.random.RandomState(seed)
    result = np.full(len(rows), -1, dtype=int)
    offset = 0
    for group in group_names(rows):
        members = sorted([i for i, r in enumerate(rows) if r["group"] == group], key=lambda i: rows[i]["id"])
        shuffled = rng.permutation(members)
        result[shuffled] = (np.arange(len(shuffled)) + offset) % count
        offset = (offset + len(shuffled)) % count
    if (result < 0).any():
        raise ValueError("Missing fold assignments")
    return _keep_clusters_together(rows, result)


def partition(rows, seed=0):
    """Same stable largest-remainder 60/40 allocation as the preserved source."""
    rng = np.random.RandomState(seed)
    head = set()
    for group in group_names(rows):
        members = sorted([r for r in rows if r["group"] == group], key=lambda r: r["id"])
        quotas = np.array([.6, .4]) * len(members)
        counts = np.floor(quotas).astype(int)
        for i in np.argsort(-(quotas - counts), kind="stable")[:len(members) - counts.sum()]:
            counts[i] += 1
        order = rng.permutation(len(members))
        head.update(members[i]["id"] for i in order[:counts[0]])
    side = _keep_clusters_together(rows, np.array([r["id"] in head for r in rows]))
    return dict(head_training=[r for r, h in zip(rows, side) if h],
                combiner_training=[r for r, h in zip(rows, side) if not h])


def stratified_subsample(rows, size, seed=0):
    """Smaller validation set for PLAN Sect. 7: every row of the smallest group is kept, the other
    groups share the remaining size in proportion to their sizes (largest remainder). Keeping the
    smallest group whole keeps DPE's group-balanced head pools able to fill a 256-row batch at every
    nesting level (a proportional 1,199-image CelebA subsample would leave 11 blond-male rows).
    Clusters partly drawn are completed. Returns sorted indices."""
    rng = np.random.RandomState(seed)
    names = group_names(rows)
    sizes = np.array([sum(r["group"] == g for r in rows) for g in names])
    smallest = int(np.argmin(sizes))
    if size <= sizes[smallest] * len(names) or size >= sizes.sum():
        raise ValueError(f"Subsample size {size} must exceed {len(names)} x the smallest group ({sizes[smallest]})")
    others = np.arange(len(names)) != smallest
    quotas = np.zeros(len(names))
    quotas[others] = sizes[others] / sizes[others].sum() * (size - sizes[smallest])
    counts = np.floor(quotas).astype(int)
    counts[smallest] = sizes[smallest]
    for i in np.argsort(-(quotas - counts), kind="stable")[:size - counts.sum()]:
        counts[i] += 1
    if (counts[others] < sizes[smallest] * 0.5).any():
        raise ValueError(f"Subsample leaves a group much smaller than the kept smallest group: {dict(zip(names, counts))}")
    chosen = set()
    for g, k in zip(names, counts):
        members = sorted([i for i, r in enumerate(rows) if r["group"] == g], key=lambda i: rows[i]["id"])
        chosen.update(np.asarray(members)[rng.permutation(len(members))[:k]].tolist())
    clusters = {rows[i].get("cluster", rows[i]["id"]) for i in chosen}
    return np.array(sorted(i for i, r in enumerate(rows) if r.get("cluster", r["id"]) in clusters))


def load_split(path, dim):
    with np.load(path, allow_pickle=False) as value:
        missing = [k for k in FEATURE_KEYS if k not in value]
        if missing:
            raise ValueError(f"{path}: missing arrays {missing}")
        out = {k: value[k] for k in FEATURE_KEYS}
    n = len(out["ids"])
    if out["features"].shape != (n, dim) or not np.isfinite(out["features"]).all():
        raise ValueError(f"{path}: expected finite features [{n}, {dim}]")
    if len(set(out["ids"].astype(str))) != n:
        raise ValueError(f"{path}: duplicate ids")
    for k in ("y", "a", "cluster"):
        if out[k].shape != (n,):
            raise ValueError(f"{path}: misaligned {k}")
    return out


class Source:
    """One backbone: validation and test features, labels and groups.

    spec: dict(id, dataset, seed, features=dict(val=path, test=path), feature_dim,
               [val_subsample=int, val_subsample_seed=int]).
    """

    def __init__(self, spec):
        self.spec = spec
        dim = spec["feature_dim"]
        val, test = load_split(spec["features"]["val"], dim), load_split(spec["features"]["test"], dim)
        rows = [dict(id=str(i), y=int(y), group=f"{int(y)}{int(a)}", cluster=str(c))
                for i, y, a, c in zip(val["ids"], val["y"], val["a"], val["cluster"])]
        keep = np.arange(len(rows))
        if spec.get("val_subsample"):
            keep = stratified_subsample(rows, int(spec["val_subsample"]), spec.get("val_subsample_seed", 0))
        self.rows = [rows[i] for i in keep]
        self._val = val["features"][keep]
        self._test = test["features"]
        self.val_y = np.array([r["y"] for r in self.rows])
        self.val_g = np.array([r["group"] for r in self.rows])
        self.test_y = test["y"].astype(int)
        self.test_g = np.array([f"{int(y)}{int(a)}" for y, a in zip(test["y"], test["a"])])
        self.test_ids = test["ids"].astype(str)
        if set(r["id"] for r in self.rows) & set(self.test_ids):
            raise ValueError("Validation/test overlap")
        self.groups = group_names(self.rows)
        if set(self.test_g) - set(self.groups):
            raise ValueError("Test has groups that validation lacks")
        self.classes = int(spec.get("classes", 2))
        if set(self.val_y) != set(range(self.classes)):
            raise ValueError("Validation must contain every class")

In [ ]:
%%writefile "{CODE}/xdpe/datasets.py"
"""Per-dataset settings fixed in PLAN.md Sect. 4 (head settings from dpe4subpop train_all_pe.sh)."""

COMMON_HEAD = dict(heads=15, epochs=20, batch_size=256, momentum=0.9, entropic_scale=30.0, wd_weight=10.0)

DATASETS = {
    # Reference recipe; used only by the equivalence test against the Waterbirds code.
    "Waterbirds": dict(classes=2, feature_dim=2048, head=dict(COMMON_HEAD, lr=1e-3, cov_reg=5e5)),
    "CelebA": dict(classes=2, feature_dim=2048, head=dict(COMMON_HEAD, lr=5e-4, cov_reg=5e5)),
    "CivilCommentsFine": dict(classes=2, feature_dim=768, head=dict(COMMON_HEAD, lr=1e-4, cov_reg=1e5)),
    "MultiNLI": dict(classes=3, feature_dim=768, head=dict(COMMON_HEAD, lr=1e-4, cov_reg=5e6)),
}

# Calibration settings shared with the Waterbirds study (configs/dpe_independent/study.yaml).
CALIBRATION = dict(steps=400, learning_rate=0.01, hidden_units=8, seed_offset=10000)


def head_settings(dataset):
    return dict(DATASETS[dataset]["head"])


def config(dataset):
    """The cfg dict consumed by the ported fitting code (same keys as study.yaml)."""
    return dict(CALIBRATION, head_training=head_settings(dataset))

In [ ]:
%%writefile "{CODE}/xdpe/heads.py"
"""DPE prototype heads on frozen features: training and probabilities.

Ported from src/methods/dpe_head_reweighting/{train_heads,ensemble}.py and
src/methods/dpe_independent/geometry.py with K classes and G groups. One prototype per class
per head; heads trained in sequence on group-balanced subsets (n = smallest group, groups
interleaved in decreasing size order), covariance penalty decaying by stage, and for each head
the epoch with the highest WGA on its own training pool.
"""
import random

import numpy as np
import torch
from scipy.special import softmax
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset


class Head(nn.Module):
    def __init__(self, dim, classes=2):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(classes, dim).normal_(0, .01))
        self.distance_scale = nn.Parameter(torch.ones(1))

    def forward(self, x):
        return -self.distance_scale.abs() * torch.cdist(
            F.normalize(x, dim=-1), F.normalize(self.prototypes, dim=-1),
            p=2, compute_mode='donot_use_mm_for_euclid_dist')


def standardize(raw):
    """Per-row z-score in float32 (ddof 0), as the preserved notebook."""
    raw = np.asarray(raw)
    if raw.ndim != 2 or not len(raw) or raw.shape[1] < 2 or raw.dtype.kind != 'f' or not np.isfinite(raw).all():
        raise ValueError('Invalid raw features')
    if raw.dtype.itemsize < 4:
        raw = raw.astype(np.float32)
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        sd = raw.std(axis=1, keepdims=True)
        if (sd == 0).any():
            raise ValueError('Zero-variance feature row')
        normalized = np.asarray((raw - raw.mean(axis=1, keepdims=True)) / sd, dtype=np.float32)
    if not np.isfinite(sd).all() or not np.isfinite(normalized).all():
        raise ValueError('Feature standardization overflow')
    return torch.from_numpy(normalized)


def pool_wga(p, y, groups, names):
    correct = p.argmax(-1) == y
    return min(float(correct[groups == g].mean()) for g in names)


def train(head_raw, head_labels, head_groups, settings, seed, classes=2, device='cpu'):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    x = standardize(head_raw)
    y, groups = np.asarray(head_labels), np.asarray(head_groups)
    names = tuple(sorted(set(groups)))
    if set(y) != set(range(classes)):
        raise ValueError('Head-training pool lacks a class')
    pools = [np.flatnonzero(groups == g) for g in names]
    order = sorted(range(len(names)), key=lambda i: (-len(pools[i]), i))
    n = min(map(len, pools))
    if len(names) * n < settings['batch_size']:
        raise ValueError(f'Balanced pool of {len(names)}x{n} rows gives zero batches of {settings["batch_size"]}')
    prototypes, scales, epochs = [], [], []
    for stage in range(1, settings['heads'] + 1):
        idx = np.stack([np.random.permutation(pools[i])[:n] for i in order], axis=1).ravel()
        loader = DataLoader(TensorDataset(x[idx], torch.tensor(y[idx], dtype=torch.long)),
                            batch_size=settings['batch_size'], shuffle=True, drop_last=True)
        head = Head(x.shape[1], classes).to(device)
        opt = torch.optim.SGD(head.parameters(), lr=settings['lr'], momentum=settings['momentum'])
        cov_reg = settings['cov_reg'] / (1 + .05 * max(0, stage - 2))
        best_wga, best = -1., None
        for epoch in range(1, settings['epochs'] + 1):
            for feats, labels in loader:
                opt.zero_grad()
                logits = head(feats.to(device))
                log_probs = F.log_softmax(settings['entropic_scale'] * logits, dim=1)
                loss = -log_probs[torch.arange(len(labels), device=device), labels.to(device)].mean()
                p = head.prototypes[:, None]
                loss = loss + torch.einsum('ijk,ilk->ijl', p, p).squeeze().mean() * settings['wd_weight']
                if prototypes and cov_reg > 0:
                    ps = torch.cat([p] + [v.to(device)[:, None] for v in prototypes], dim=1)
                    cov = torch.einsum('ijk,ilk->ijl', ps, ps) / (x.shape[1] - 1)
                    loss = loss + cov[:, 0, 1:].sum(1).div(ps.shape[1]).mean().abs() * cov_reg
                if not torch.isfinite(loss):
                    raise ValueError('Nonfinite training loss')
                loss.backward()
                opt.step()
            with torch.no_grad():
                probabilities = torch.cat([head(b.to(device)).softmax(-1).cpu() for b in x.split(256)]).numpy()
            wga = pool_wga(probabilities, y, groups, names)
            if wga >= best_wga:
                best_wga = wga
                best = (head.prototypes.detach().cpu().clone(), head.distance_scale.detach().cpu().clone(), epoch)
        prototypes.append(best[0])
        scales.append(best[1].squeeze())
        epochs.append(best[2])
    return dict(prototypes=torch.stack(prototypes), distance_scales=torch.stack(scales), selected_epochs=epochs)


def probabilities(heads, raw, batch_size=256):
    """[H, N, K] float64 probabilities, rounded through float32 like the saved caches.

    Distances in float64 on the float32 row z-score (geometry.distances), scores
    -|scale| * distance, softmax over classes."""
    x = standardize(raw).double()
    distances = []
    with torch.inference_mode():
        for prototypes in heads["prototypes"]:
            p = F.normalize(prototypes.to(dtype=torch.float64), dim=-1)
            distances.append(torch.cat([
                torch.cdist(F.normalize(b, dim=-1), p, p=2, compute_mode="donot_use_mm_for_euclid_dist")
                for b in x.split(batch_size)]).numpy())
    d = np.stack(distances)
    s = np.abs(heads["distance_scales"].numpy().astype(np.float64))
    if not np.isfinite(d).all() or (d < 0).any() or not np.isfinite(s).all():
        raise ValueError("Nonfinite distances or scales")
    z = -s[:, None, None] * d
    return softmax(z, axis=-1).astype(np.float32).astype(np.float64)

In [ ]:
%%writefile "{CODE}/xdpe/multiclass.py"
"""K-class (K >= 3) versions of the calibrators, as written in PLAN.md Sect. 9.

Inputs are mean head probabilities [N, K] (bias, affine) or per-head probabilities
[N, 15, K] (linear residual). With s = log(clip(mean probability)):

  bias    : logits = s + b                        penalty  lambda * ||b||^2
  affine  : logits = a * s + b                    penalty  lambda * ((a-1)^2 + ||b||^2)
  linear  : affine + W vec(q_j - mean q)          penalty  + lambda * mean(W^2), W starts at 0

All fitted by the same 400 full-batch Adam steps (lr 0.01) on group-balanced ("average") or
worst-group ("maximum") cross-entropy. The binary offset-only map uses a bounded scalar
search instead; with K >= 3 there is no scalar threshold, so the exact-WGA threshold arms are
not defined. Decisions are the argmax.
"""
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

from .calibrate import cpu_randomness, group_losses, names_of


def log_mean(p):
    return np.log(np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0))


class MulticlassMap(nn.Module):
    def __init__(self, kind, classes, heads=15):
        super().__init__()
        if kind not in ("bias", "affine", "linear"):
            raise ValueError(kind)
        self.kind = kind
        self.b = nn.Parameter(torch.zeros(classes, dtype=torch.float64))
        if kind != "bias":
            self.a = nn.Parameter(torch.ones((), dtype=torch.float64))
        if kind == "linear":
            self.correction = nn.Linear(heads * classes, classes, bias=False).double()
            nn.init.zeros_(self.correction.weight)

    def logits(self, x):
        if self.kind == "linear":
            mean = x.mean(1)
            s = torch.log(mean.clamp(1e-12, 1.0))
            disagreement = (x - mean[:, None, :]).reshape(len(x), -1)
            return self.a * s + self.b + self.correction(disagreement)
        s = torch.log(x.clamp(1e-12, 1.0))
        return (self.a * s if hasattr(self, "a") else s) + self.b

    def regularizer(self):
        result = self.b.square().sum()
        if hasattr(self, "a"):
            result = result + (self.a - 1).square()
        if hasattr(self, "correction"):
            result = result + self.correction.weight.square().mean()
        return result


def fit(x, y, groups, kind, setting, cfg, seed):
    y, groups = np.asarray(y), np.asarray(groups)
    names = names_of(groups)
    xt = torch.tensor(np.asarray(x), dtype=torch.float64)
    target = torch.tensor(y, dtype=torch.long)
    g = torch.tensor([names.index(v) for v in groups])
    classes = xt.shape[-1]
    loss_name, penalty = setting
    with cpu_randomness(seed):
        model = MulticlassMap(kind, classes, xt.shape[1] if kind == "linear" else 15)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
        for step in range(cfg["steps"] + 1):
            per_example = F.cross_entropy(model.logits(xt), target, reduction="none")
            losses = group_losses(per_example, g, len(names))
            objective = (losses.mean() if loss_name == "average" else losses.max()) + penalty * model.regularizer()
            if not torch.isfinite(objective):
                raise ValueError("Nonfinite multiclass fit")
            if step == cfg["steps"]:
                break
            optimizer.zero_grad(set_to_none=True)
            objective.backward()
            optimizer.step()
    return dict(kind=kind, classes=classes, heads=xt.shape[1] if kind == "linear" else 15,
                state={k: v.detach().tolist() for k, v in model.state_dict().items()}, seed=seed)


def predict(model, x):
    with cpu_randomness(0):
        m = MulticlassMap(model["kind"], model["classes"], model["heads"])
        m.load_state_dict({k: torch.tensor(v, dtype=torch.float64) for k, v in model["state"].items()}, strict=True)
        with torch.inference_mode():
            p = m.logits(torch.tensor(np.asarray(x), dtype=torch.float64)).softmax(-1).numpy()
    if not np.isfinite(p).all():
        raise ValueError("Nonfinite multiclass probabilities")
    return p


def balanced_nll(prob, y, groups):
    p = np.clip(np.asarray(prob, dtype=np.float64)[np.arange(len(y)), y], 1e-12, 1.0)
    loss = -np.log(p)
    return float(np.mean([loss[groups == k].mean() for k in names_of(groups)]))


def inner_score(prob, y, groups):
    from .calibrate import group_accuracy
    acc = group_accuracy(np.asarray(prob).argmax(-1), y, groups)
    return dict(wga=min(acc.values()) * 100, nll=balanced_nll(prob, y, groups))

In [ ]:
%%writefile "{CODE}/xdpe/nested.py"
"""Strictly nested validation-only DPE post-hoc study on any frozen-feature dataset.

Port of experiments/dpe_nested_corrected/code/corrected_nested.py (protocol
strict-nested-postmean-v2) without DFR-lite (PLAN.md Sect. 4). Same arms, settings, seeds,
selection rules, cross-fit and inner folds, 60/40 pooled controls and 75-head bag.

    python -m xdpe.nested discover --roots DIR [DIR ...] --dataset CelebA --out OUT [--val-subsample 1199]
    python -m xdpe.nested run --out OUT [--splits 0 1 ...] [--workers 4] [--time-limit-hours 11] [--smoke]

`--smoke` trains 3-head ensembles for 2 epochs and never scores the test set (PLAN Sect. 8.4);
use a separate --out for it. `--no-test` runs the full recipe without scoring the test set
(timing, PLAN step 0.4); its checkpoints are reused when the same --out is later run without
the flag. Test predictions are scored only after every fit and selection of a cell is written
to fit_complete.json, as in the Waterbirds run.
"""
import os

for _key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_key, "1")

import argparse
import hashlib
import json
import math
import platform
import sys
import tempfile
import time
import traceback
from pathlib import Path

import numpy as np

from . import calibrate as cal
from . import multiclass as mc
from .data import Source, folds, partition, sha256
from .datasets import DATASETS, config
from .heads import probabilities, train

HERE = Path(__file__).resolve().parent
PROTOCOL = "xdpe-strict-nested-postmean-v2"
INNER_FOLDS = 5
CROSSFIT_FOLDS = 5
BAG_ENSEMBLES = 5
SPLIT_SEED_STRIDE = 1000
FAMILIES = ("crossfit_affine", "crossfit_bias_foldwise", "crossfit_postmean_bias",
            "fullval_affine_oof", "fullval_postmean_bias")
SMOKE_HEADS = dict(heads=3, epochs=2)


# ----------------------------------------------------------------------------- io

def log(*parts):
    print(f"[{time.strftime('%H:%M:%S')}]", *parts, flush=True)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=".pending-")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(value, f, indent=1, sort_keys=True)
        os.replace(tmp, path)
    finally:
        Path(tmp).unlink(missing_ok=True)


def read_json(path):
    return json.loads(Path(path).read_text())


def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def code_identity():
    files = sorted(p for p in HERE.glob("*.py"))
    return digest_json({p.name: sha256(p) for p in files})


def bind_manifest(path, value):
    path = Path(path)
    if path.exists():
        if read_json(path) != value:
            raise ValueError(f"Cache identity mismatch: {path}. Use a fresh output directory.")
    else:
        write_json(path, value)


def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(dir=path.parent, prefix=".pending-", suffix=".npz")
    try:
        with os.fdopen(fd, "wb") as f:
            np.savez_compressed(f, **arrays)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)


class BudgetReached(Exception):
    pass


def check_budget(deadline):
    if time.time() >= deadline:
        raise BudgetReached("Time budget reached between checkpointed head fits")


# ----------------------------------------------------------------------------- binary / K-class switch

def binary(source):
    return source.classes == 2


def summarise_heads(source, q):
    """[H, N, K] head probabilities -> the ensemble score kept in caches."""
    mean = q.mean(0)
    return mean[:, 1] if binary(source) else mean


def decide(p):
    p = np.asarray(p)
    return p > .5 if p.ndim == 1 else p.argmax(-1)


def score(p, y, g):
    return cal.inner_score(p, y, g) if np.asarray(p).ndim == 1 else mc.inner_score(p, y, g)


def fit_map(source, x, y, g, kind, setting, cfg, seed):
    if binary(source):
        return cal.fit_model(x[:, None] if x.ndim == 1 else x, y, g, kind, setting, cfg, seed)
    return mc.fit(x, y, g, kind, setting, cfg, seed)


def apply_map(source, model, p, cfg):
    if binary(source):
        p = np.asarray(p)
        return cal.predict_model(model, p[:, None] if p.ndim == 1 else p, cfg)
    return mc.predict(model, p)


# ----------------------------------------------------------------------------- heads and caches

_SOURCES = {}


def get_source(spec):
    if spec["id"] not in _SOURCES:
        _SOURCES[spec["id"]] = Source(spec)
    return _SOURCES[spec["id"]]


def train_ensemble(source, indices, seed, cfg):
    indices = np.asarray(indices)
    return train(source._val[indices], source.val_y[indices], source.val_g[indices],
                 cfg["head_training"], seed, classes=source.classes)


def cached_scores(source, train_idx, val_eval_idx, test_eval, seed, cfg, directory, deadline, forbidden=()):
    tr, ve = np.asarray(train_idx, dtype=int), np.asarray(val_eval_idx, dtype=int)
    forbidden = np.asarray(forbidden, dtype=int)
    if len(set(tr)) != len(tr) or np.intersect1d(tr, ve).size or np.intersect1d(tr, forbidden).size:
        raise ValueError("Head training overlaps its held-out validation rows")
    stage = Path(directory)
    stage.mkdir(parents=True, exist_ok=True)
    bind_manifest(stage / "indices.json", dict(train_idx=tr.tolist(), val_eval_idx=ve.tolist(),
                                               forbidden_idx=forbidden.tolist(), test_eval=bool(test_eval), seed=int(seed)))
    cache, marker = stage / "scores.npz", stage / "complete.json"
    tail = () if binary(source) else (source.classes,)
    if cache.exists() and marker.exists():
        if read_json(marker)["sha256"] != sha256(cache):
            raise ValueError(f"Prediction checkpoint checksum mismatch: {cache}")
        with np.load(cache, allow_pickle=False) as data:
            val, test = data["val"], data["test"]
        if val.shape != (len(ve), *tail) or test.shape != (((len(source._test), *tail)) if test_eval else (0, *tail)):
            raise ValueError("Prediction checkpoint shape mismatch")
        return val, test
    check_budget(deadline)
    heads = train_ensemble(source, tr, seed, cfg)
    val = summarise_heads(source, probabilities(heads, source._val[ve])) if len(ve) else np.empty((0, *tail))
    test = summarise_heads(source, probabilities(heads, source._test)) if test_eval else np.empty((0, *tail))
    if not np.isfinite(val).all() or not np.isfinite(test).all():
        raise ValueError("Nonfinite head probabilities")
    atomic_npz(cache, val=val, test=test)
    write_json(marker, dict(sha256=sha256(cache), selected_epochs=heads["selected_epochs"]))
    return val, test


def build_context(source, pool, evaluation, test_eval, fold_seed, head_seed, cfg, directory, deadline, forbidden=()):
    pool, evaluation = np.asarray(pool, dtype=int), np.asarray(evaluation, dtype=int)
    if np.intersect1d(pool, evaluation).size or np.intersect1d(pool, forbidden).size:
        raise ValueError("Context pool overlaps an inner evaluation fold")
    rows = [source.rows[i] for i in pool]
    assignment = folds(rows, CROSSFIT_FOLDS, fold_seed)
    tail = () if binary(source) else (source.classes,)
    oof = np.full((len(pool), *tail), np.nan)
    fold_eval = []
    for f in range(CROSSFIT_FOLDS):
        tr = pool[assignment != f]
        te_local = np.flatnonzero(assignment == f)
        held = pool[te_local]
        pv, pt = cached_scores(source, tr, np.r_[held, evaluation], test_eval, head_seed, cfg,
                               Path(directory) / f"fold{f}", deadline, forbidden)
        oof[te_local] = pv[:len(held)]
        fold_eval.append(pt if test_eval else pv[len(held):])
    if not np.isfinite(oof).all():
        raise ValueError("Incomplete OOF scores")
    pv, pt = cached_scores(source, pool, evaluation, test_eval, head_seed, cfg,
                           Path(directory) / "full_refit", deadline, forbidden)
    return dict(oof=oof, folds=np.stack(fold_eval), full=pt if test_eval else pv)


# ----------------------------------------------------------------------------- calibration arms

def calibration_predictions(source, affine, bias, fold_scores, full_scores, cfg):
    """Exactly the same inference graph in selection and final test evaluation."""
    apply = lambda model, p: apply_map(source, model, p, cfg)
    return dict(crossfit_affine=np.mean([apply(affine, p) for p in fold_scores], axis=0),
                crossfit_bias_foldwise=np.mean([apply(bias, p) for p in fold_scores], axis=0),
                crossfit_postmean_bias=apply(bias, np.mean(fold_scores, axis=0)),
                fullval_affine_oof=apply(affine, full_scores),
                fullval_postmean_bias=apply(bias, full_scores))


def strict_inner_scores(source, split, cfg, base, fit_seed, directory, deadline):
    n = len(source.rows)
    assignment = folds(source.rows, INNER_FOLDS, 10_000 + split)
    predictions = {family: {cal.name(s): None for s in cal.SETTINGS} for family in FAMILIES}
    filled = {family: {cal.name(s): np.zeros(n, dtype=bool) for s in cal.SETTINGS} for family in FAMILIES}
    thresholds = ("crossfit_postmean_thrwga", "fullval_postmean_thrwga") if binary(source) else ()
    threshold_predictions = {family: np.zeros(n, dtype=bool) for family in thresholds}
    threshold_logs = []
    for f in range(INNER_FOLDS):
        tr, te = np.flatnonzero(assignment != f), np.flatnonzero(assignment == f)
        context = build_context(source, tr, te, False, split, base, cfg, Path(directory) / f"inner{f}",
                                deadline, forbidden=te)
        y, g = source.val_y[tr], source.val_g[tr]
        x = context["oof"]
        for setting in cal.SETTINGS:
            affine = fit_map(source, x, y, g, "affine", setting, cfg, fit_seed)
            bias = fit_map(source, x, y, g, "bias", setting, cfg, fit_seed)
            for family, p in calibration_predictions(source, affine, bias, context["folds"], context["full"], cfg).items():
                key = cal.name(setting)
                if predictions[family][key] is None:
                    predictions[family][key] = np.full((n, *p.shape[1:]), np.nan)
                predictions[family][key][te] = p
                filled[family][key][te] = True
        if thresholds:
            threshold = cal.fit_threshold_exact(x[:, None], y, g)
            threshold_logs.append(dict(inner_fold=f, model=threshold))
            threshold_predictions["crossfit_postmean_thrwga"][te] = cal.apply_threshold(threshold, context["folds"].mean(0))
            threshold_predictions["fullval_postmean_thrwga"][te] = cal.apply_threshold(threshold, context["full"])
    scores = {}
    for family, settings in predictions.items():
        if any(not filled[family][k].all() or not np.isfinite(p).all() for k, p in settings.items()):
            raise ValueError("Incomplete inner evaluation predictions")
        scores[family] = {k: score(p, source.val_y, source.val_g) for k, p in settings.items()}
    diagnostics = {family: cal.record(p, source.val_y, source.val_g) for family, p in threshold_predictions.items()}
    return scores, dict(threshold_folds=threshold_logs, inner_threshold_performance=diagnostics)


def final_predictions(source, context, inner, cfg, fit_seed):
    y, g = source.val_y, source.val_g
    x = context["oof"]
    grid = {family: {} for family in FAMILIES}
    models = {}
    for setting in cal.SETTINGS:
        affine = fit_map(source, x, y, g, "affine", setting, cfg, fit_seed)
        bias = fit_map(source, x, y, g, "bias", setting, cfg, fit_seed)
        models[cal.name(setting)] = dict(affine=affine, bias=bias)
        for family, p in calibration_predictions(source, affine, bias, context["folds"], context["full"], cfg).items():
            grid[family][cal.name(setting)] = p
    chosen = {family: {rule: cal.choose(inner[family], rule) for rule in cal.RULES} for family in FAMILIES}
    threshold = cal.fit_threshold_exact(x[:, None], y, g) if binary(source) else None
    return grid, chosen, threshold, models


def pool_inputs(source, q):
    """[H, N, K] head probabilities -> combiner inputs: binary [N, H]; K-class [N, H, K]."""
    return q[:, :, 1].T if binary(source) else q.transpose(1, 0, 2)


def pool_fit(source, x, y, g, kind, setting, cfg, seed):
    if binary(source):
        return cal.fit_model(x, y, g, kind, setting, cfg, seed)
    return mc.fit(x if kind == "linear" else x.mean(1), y, g, kind, setting, cfg, seed)


def pool_apply(source, model, x, cfg):
    if binary(source):
        return cal.predict_model(model, x, cfg)
    return mc.predict(model, x if model["kind"] == "linear" else x.mean(1))


def pool_inner_cv(source, x, y, g, rows, kind, cfg, seed, split_seed):
    assign = folds(rows, INNER_FOLDS, 10_000 + split_seed)
    out = {}
    for setting in cal.SETTINGS:
        oof = None
        for f in range(INNER_FOLDS):
            tr, te = assign != f, assign == f
            model = pool_fit(source, x[tr], y[tr], g[tr], kind, setting, cfg, seed)
            p = pool_apply(source, model, x[te], cfg)
            if oof is None:
                oof = np.full((len(y), *p.shape[1:]), np.nan)
            oof[te] = p
        out[cal.name(setting)] = score(oof, y, g)
    return out


def pooled_predictions(source, split, cfg, base, fit_seed, directory, deadline):
    part = partition(source.rows, split)
    lookup = {r["id"]: i for i, r in enumerate(source.rows)}
    head_idx = np.array([lookup[r["id"]] for r in part["head_training"]])
    fit_idx = np.array([lookup[r["id"]] for r in part["combiner_training"]])
    if np.intersect1d(head_idx, fit_idx).size:
        raise ValueError("Pooled head/combiner overlap")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    bind_manifest(directory / "indices.json", dict(head_idx=head_idx.tolist(), fit_idx=fit_idx.tolist()))
    path, marker = directory / "head_scores.npz", directory / "complete.json"
    if path.exists() and marker.exists():
        if read_json(marker)["sha256"] != sha256(path):
            raise ValueError("Pooled head-score checksum mismatch")
        with np.load(path, allow_pickle=False) as value:
            qf, qt = value["fit"], value["test"]
    else:
        check_budget(deadline)
        heads = train_ensemble(source, head_idx, base, cfg)
        qf = probabilities(heads, source._val[fit_idx])
        qt = probabilities(heads, source._test)
        atomic_npz(path, fit=qf, test=qt)
        write_json(marker, dict(sha256=sha256(path)))
    xf, xt = pool_inputs(source, qf), pool_inputs(source, qt)
    y, g = source.val_y[fit_idx], source.val_g[fit_idx]
    rows = [source.rows[i] for i in fit_idx]
    inner, grid, chosen = {}, {}, {}
    for kind in ("affine", "linear"):
        family = "pool_" + kind
        inner[family] = pool_inner_cv(source, xf, y, g, rows, kind, cfg, fit_seed, split)
        grid[family] = {}
        for setting in cal.SETTINGS:
            model = pool_fit(source, xf, y, g, kind, setting, cfg, fit_seed)
            grid[family][cal.name(setting)] = pool_apply(source, model, xt, cfg)
        chosen[family] = {rule: cal.choose(inner[family], rule) for rule in cal.RULES}
    return summarise_heads(source, qt), grid, inner, chosen


# ----------------------------------------------------------------------------- one cell

def run_task(task):
    spec, split, out_dir, cfg, protocol_id, deadline, score_test = task
    started = time.time()
    out = Path(out_dir)
    cell = out / "results" / spec["id"] / f"split{split:02d}.json"
    checkpoints = out / "checkpoints" / spec["id"] / f"split{split:02d}"
    error = out / "errors" / f"{spec['id']}-split{split:02d}.txt"
    try:
        import torch
        torch.set_num_threads(1)
        source = get_source(spec)
        source_id = dict(id=spec["id"], dataset=spec["dataset"], seed=spec["seed"],
                         val_subsample=spec.get("val_subsample"),
                         features={k: sha256(v) for k, v in spec["features"].items()})
        fingerprint = digest_json(dict(protocol=protocol_id, source=source_id, split=split))
        bind_manifest(checkpoints / "identity.json", dict(fingerprint=fingerprint, source=source_id))
        if cell.exists():
            if read_json(cell).get("fingerprint") != fingerprint:
                raise ValueError("Completed result was produced by another protocol/source")
            error.unlink(missing_ok=True)
            return spec["id"], split, "cached", 0.
        base = spec["seed"] + SPLIT_SEED_STRIDE * split
        fit_seed = spec["seed"] + cfg["seed_offset"]
        inner_path = checkpoints / "inner_scores.json"
        if inner_path.exists():
            saved = read_json(inner_path)
            inner, diagnostics = saved["scores"], saved["diagnostics"]
        else:
            inner, diagnostics = strict_inner_scores(source, split, cfg, base, fit_seed, checkpoints, deadline)
            write_json(inner_path, dict(scores=inner, diagnostics=diagnostics))
        context = build_context(source, np.arange(len(source.rows)), np.array([], dtype=int),
                                True, split, base, cfg, checkpoints / "final", deadline)
        grid_prob, chosen, threshold, models = final_predictions(source, context, inner, cfg, fit_seed)
        prob_arms = dict(fullval_uniform=context["full"], crossfit_uniform=context["folds"].mean(0))
        full_bag = [context["full"]]
        for k in range(1, BAG_ENSEMBLES):
            _, p = cached_scores(source, np.arange(len(source.rows)), np.array([], dtype=int), True,
                                 base + k, cfg, checkpoints / f"bag{k}", deadline)
            full_bag.append(p)
        prob_arms["bag75_uniform"] = np.mean(full_bag, axis=0)
        selected = {}
        for family in FAMILIES:
            prob_arms[family + "_fixed"] = grid_prob[family][cal.name(cal.FIXED)]
            selected[family + "_fixed"] = cal.name(cal.FIXED)
            for rule in cal.RULES:
                arm = family + "_sel_" + rule
                selected[arm] = chosen[family][rule]
                prob_arms[arm] = grid_prob[family][chosen[family][rule]]
        pool_uniform, pool_grid, pool_inner, pool_chosen = pooled_predictions(
            source, split, cfg, base, fit_seed, checkpoints / "pooled", deadline)
        prob_arms["pool_uniform"] = pool_uniform
        inner.update(pool_inner)
        chosen.update(pool_chosen)
        grid_prob.update(pool_grid)
        for family in pool_grid:
            prob_arms[family + "_fixed"] = pool_grid[family][cal.name(cal.FIXED)]
            selected[family + "_fixed"] = cal.name(cal.FIXED)
            for rule in cal.RULES:
                arm = family + "_sel_" + rule
                selected[arm] = pool_chosen[family][rule]
                prob_arms[arm] = pool_grid[family][selected[arm]]
        # Durable fit/selection record, written before any test label is read.
        write_json(checkpoints / "fit_complete.json", dict(chosen=chosen, threshold=threshold,
                   models=models, fingerprint=fingerprint))
        if not score_test:
            write_json(out / "smoke" / spec["id"] / f"split{split:02d}.json",
                       dict(protocol=PROTOCOL, fingerprint=fingerprint, chosen=chosen, inner=inner,
                            arms=sorted(prob_arms) + (["crossfit_postmean_thrwga", "fullval_postmean_thrwga"] if threshold else []),
                            seconds=time.time() - started))
            error.unlink(missing_ok=True)
            return spec["id"], split, "smoke-done", time.time() - started
        ty, tg = source.test_y, source.test_g
        np.savez_compressed(checkpoints / "test_decisions.npz",
                            **{arm: decide(p) for arm, p in prob_arms.items()})
        arms = {arm: cal.record(decide(p), ty, tg, **({"selected": selected[arm]} if arm in selected else {}))
                for arm, p in prob_arms.items()}
        if threshold is not None:
            arms["crossfit_postmean_thrwga"] = cal.record(cal.apply_threshold(threshold, context["folds"].mean(0)), ty, tg, **threshold)
            arms["fullval_postmean_thrwga"] = cal.record(cal.apply_threshold(threshold, context["full"]), ty, tg, **threshold)
        grid = {family: {k: cal.record(decide(p), ty, tg) for k, p in settings.items()}
                for family, settings in grid_prob.items()}
        write_json(cell, dict(protocol=PROTOCOL, fingerprint=fingerprint, source=spec["id"], dataset=spec["dataset"],
                              seed=spec["seed"], split=split, val_subsample=spec.get("val_subsample"),
                              arms=arms, grid=grid, inner=inner, diagnostics=diagnostics,
                              seconds=time.time() - started))
        error.unlink(missing_ok=True)
        return spec["id"], split, "done", time.time() - started
    except BudgetReached:
        return spec["id"], split, "deferred", time.time() - started
    except Exception:
        error.parent.mkdir(parents=True, exist_ok=True)
        error.write_text(traceback.format_exc())
        return spec["id"], split, "ERROR", time.time() - started


# ----------------------------------------------------------------------------- commands

def protocol_config(dataset, smoke):
    cfg = config(dataset)
    if smoke:
        cfg["head_training"].update(SMOKE_HEADS)
    return cfg


def prepare_protocol(out, dataset, smoke):
    import scipy
    import torch
    cfg = protocol_config(dataset, smoke)
    identity = dict(protocol=PROTOCOL, dataset=dataset, smoke=bool(smoke), code_sha256=code_identity(), config=cfg,
                    inner_folds=INNER_FOLDS, crossfit_folds=CROSSFIT_FOLDS, bag_ensembles=BAG_ENSEMBLES,
                    settings=cal.SETTINGS, fixed=cal.FIXED, primary_rule="nll", secondary_rule="wga",
                    python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
                    torch=torch.__version__)
    identity = json.loads(json.dumps(identity))
    bind_manifest(Path(out) / "protocol.json", identity)
    return cfg, digest_json(identity)


def cmd_discover(a):
    """Find complete backbones written by train_backbone.py (backbone.json + features/*.npz)."""
    found = {}
    for root in a.roots:
        for manifest in sorted(Path(root).rglob("backbone.json")):
            m = read_json(manifest)
            if m.get("dataset") != a.dataset or m.get("status") != "complete":
                continue
            base = manifest.parent
            features = dict(val=str((base / "features/val.npz").resolve()), test=str((base / "features/test.npz").resolve()))
            if not all(Path(p).is_file() for p in features.values()):
                log("skip (missing features):", base)
                continue
            hashes = {k: sha256(v) for k, v in features.items()}
            for k, v in hashes.items():
                if m["features_sha256"][k] != v:
                    raise ValueError(f"{base}: feature cache differs from its manifest")
            sid = f"{a.dataset}-seed{m['seed']}" + (f"-val{a.val_subsample}" if a.val_subsample else "")
            spec = dict(id=sid, dataset=a.dataset, seed=int(m["seed"]), features=features,
                        feature_dim=DATASETS[a.dataset]["feature_dim"], classes=DATASETS[a.dataset]["classes"],
                        checkpoint_sha256=m["checkpoint_sha256"], features_sha256=hashes, manifest=str(manifest.resolve()))
            if a.val_subsample:
                spec.update(val_subsample=a.val_subsample, val_subsample_seed=0)
            if sid in found and found[sid]["features_sha256"] != hashes:
                raise ValueError(f"Two different backbones claim {sid}")
            found.setdefault(sid, spec)
    specs = sorted(found.values(), key=lambda s: s["seed"])
    write_json(Path(a.out) / "sources.json", dict(dataset=a.dataset, sources=specs))
    log(f"found {len(specs)} {a.dataset} backbones:", [s["id"] for s in specs])
    return 0 if specs else 1


def cmd_run(a):
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing
    if a.workers < 1 or not math.isfinite(a.time_limit_hours) or a.time_limit_hours <= 0 or not a.splits:
        raise ValueError("Positive workers/time budget and at least one split are required")
    sources = read_json(Path(a.out) / "sources.json")
    specs = [s for s in sources["sources"] if not a.only or s["id"] in a.only]
    if not specs:
        raise ValueError("No selected sources")
    if len(set(a.splits)) != len(a.splits) or any(s < 0 or s > 9 for s in a.splits):
        raise ValueError("Choose distinct split seeds in 0..9")
    cfg, protocol_id = prepare_protocol(a.out, sources["dataset"], a.smoke)
    deadline = time.time() + a.time_limit_hours * 3600
    score_test = not (a.smoke or a.no_test)
    tasks = [(s, split, str(a.out), cfg, protocol_id, deadline, score_test) for split in a.splits for s in specs]
    status_counts = {}
    with ProcessPoolExecutor(max_workers=a.workers, mp_context=multiprocessing.get_context("spawn")) as pool:
        queue, running = iter(tasks), {}

        def submit():
            while len(running) < a.workers and time.time() < deadline:
                task = next(queue, None)
                if task is None:
                    break
                running[pool.submit(run_task, task)] = task
        submit()
        while running:
            future = next(as_completed(running))
            running.pop(future)
            sid, split, status, seconds = future.result()
            status_counts[status] = status_counts.get(status, 0) + 1
            log(status, sid, f"split{split:02d}", f"{seconds / 60:.1f} min")
            submit()
    folder = "results" if score_test else "smoke"
    complete = sum((Path(a.out) / folder / s["id"] / f"split{split:02d}.json").exists() for split in a.splits for s in specs)
    write_json(Path(a.out) / "completion.json", dict(protocol=PROTOCOL, requested=len(tasks), completed=complete,
               splits=a.splits, sources=[s["id"] for s in specs], statuses=status_counts, smoke=bool(a.smoke),
               test_scored=score_test))
    log(f"Completed {complete}/{len(tasks)} requested cells; stage checkpoints retained.")
    return 1 if status_counts.get("ERROR") else (0 if complete == len(tasks) else 3)


def parser():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="command", required=True)
    d = sub.add_parser("discover")
    d.add_argument("--roots", nargs="+", required=True)
    d.add_argument("--dataset", required=True, choices=sorted(DATASETS))
    d.add_argument("--out", required=True, type=Path)
    d.add_argument("--val-subsample", type=int, default=0)
    r = sub.add_parser("run")
    r.add_argument("--out", required=True, type=Path)
    r.add_argument("--only", nargs="*")
    r.add_argument("--splits", type=int, nargs="+", default=list(range(10)))
    r.add_argument("--workers", type=int, default=4)
    r.add_argument("--time-limit-hours", type=float, default=11.0)
    r.add_argument("--smoke", action="store_true")
    r.add_argument("--no-test", action="store_true")
    return p


if __name__ == "__main__":
    args = parser().parse_args()
    sys.exit({"discover": cmd_discover, "run": cmd_run}[args.command](args))

In [ ]:
%%writefile "{CODE}/summarize.py"
"""Declared analysis for the cross-dataset test (PLAN.md Sect. 5-7). Do not edit after the freeze.

    python summarize.py OUT [OUT ...] --report REPORT_DIR

Each OUT is one `xdpe.nested run` output folder (one dataset, or one dataset with the
validation set subsampled). For every OUT: per backbone, average the paired test-WGA
difference over split seeds (pairs share a split seed); then mean, pointwise 95% t interval
over backbones and the number of backbones with a positive difference. Official test labels.

Decision rule (PLAN Sect. 5): supported if mean > 0 and the interval excludes 0; otherwise
not supported. Fewer than 3 backbones: no interval, "not assessable".
"""
import argparse
import csv
import hashlib
import json
import math
from pathlib import Path

import numpy as np

REFERENCE = "fullval_uniform"
CONTRASTS = [  # (id, arm, control, role)
    ("H1", "crossfit_affine_sel_nll", REFERENCE, "primary on CelebA: cross-fitted recalibration"),
    ("H2", "fullval_affine_oof_sel_nll", REFERENCE, "out-of-fold map applied to full-validation DPE"),
    ("H3", "pool_linear_sel_nll", REFERENCE, "linear residual on head disagreements"),
    ("H4", "pool_linear_sel_nll", "pool_affine_sel_nll", "head information beyond scalar recalibration"),
]
GROUP_ARMS = [REFERENCE, "crossfit_affine_sel_nll", "fullval_affine_oof_sel_nll", "pool_linear_sel_nll"]
EXPECTED_SPLITS = list(range(10))  # declared split seeds; nested_kaggle.ipynb runs exactly these (PLAN Sect. 4)
PRIMARY = ("CelebA", "H1")


def t_interval(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = float(values.mean()) if n else float("nan")
    if n < 3:
        return mean, float("nan"), float("nan")
    from scipy import stats
    half = stats.t.ppf(.975, n - 1) * values.std(ddof=1) / math.sqrt(n)
    return mean, mean - half, mean + half


def decision(mean, lo, hi):
    if math.isnan(lo):
        return "not assessable (fewer than 3 backbones)"
    return "supported" if mean > 0 and lo > 0 else "not supported"


def fmt(v, digits=2, sign=False):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "n/a"
    return f"{v:+.{digits}f}" if sign else f"{v:.{digits}f}"


def table(header, rows):
    out = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] + ["---:"] * (len(header) - 1)) + " |"]
    return "\n".join(out + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])


def load(out):
    out = Path(out)
    protocol = json.loads((out / "protocol.json").read_text())
    sources = json.loads((out / "sources.json").read_text())
    cells = [json.loads(p.read_text()) for p in sorted((out / "results").glob("*/split*.json"))]
    if any(c["protocol"] != "xdpe-strict-nested-postmean-v2" for c in cells):
        raise ValueError("Unexpected protocol in results")
    if protocol.get("smoke"):
        raise ValueError(f"{out} is a smoke run; it has no test scores")
    return protocol, sources, cells


def paired(cells, arm, control):
    """{source: mean over shared splits of (arm - control) WGA}."""
    by = {}
    for c in cells:
        if arm in c["arms"] and control in c["arms"]:
            by.setdefault(c["source"], []).append(c["arms"][arm]["wga"] - c["arms"][control]["wga"])
    return {s: float(np.mean(v)) for s, v in sorted(by.items())}


def per_source_mean(cells, arm, key="wga"):
    by = {}
    for c in cells:
        if arm in c["arms"]:
            by.setdefault(c["source"], []).append(c["arms"][arm][key])
    return {s: float(np.mean(v)) for s, v in sorted(by.items())}


def erm_reference(sources):
    """ERM backbone test WGA from each backbone manifest, if present."""
    values = {}
    for spec in sources["sources"]:
        try:
            m = json.loads(Path(spec["manifest"]).read_text())
            values[spec["id"]] = m["erm_metrics"]["test"]["wga"] * 100
        except (KeyError, OSError, ValueError):
            pass
    return values


def analyse(out):
    protocol, sources, cells = load(out)
    dataset = protocol["dataset"]
    subsample = sources["sources"][0].get("val_subsample") if sources["sources"] else None
    label = dataset + (f" (validation subsampled to {subsample})" if subsample else "")
    planned = [s["id"] for s in sources["sources"]]
    present = {(c["source"], c["split"]) for c in cells}
    missing = [(s, k) for s in planned for k in EXPECTED_SPLITS if (s, k) not in present]
    result = dict(out=str(out), dataset=dataset, label=label, backbones=planned, cells=len(cells),
                  missing=missing, contrasts=[], arms={}, groups={}, selection={})
    for cid, arm, control, role in CONTRASTS:
        d = paired(cells, arm, control)
        mean, lo, hi = t_interval(list(d.values()))
        result["contrasts"].append(dict(id=cid, arm=arm, control=control, role=role, mean=mean, lo=lo, hi=hi,
                                        positive=sum(v > 0 for v in d.values()), n=len(d), per_source=d,
                                        decision=decision(mean, lo, hi),
                                        primary=(dataset, cid) == PRIMARY and not subsample))
    arms = sorted({a for c in cells for a in c["arms"]})
    for arm in arms:
        wga = per_source_mean(cells, arm)
        oa = per_source_mean(cells, arm, "overall")
        d = paired(cells, arm, REFERENCE)
        mean, lo, hi = t_interval(list(d.values()))
        result["arms"][arm] = dict(wga=float(np.mean(list(wga.values()))), overall=float(np.mean(list(oa.values()))),
                                   delta=mean, lo=lo, hi=hi, positive=sum(v > 0 for v in d.values()), n=len(d))
    for arm in GROUP_ARMS:
        rows = [c["arms"][arm] for c in cells if arm in c["arms"]]
        if rows:
            names = sorted(rows[0]["groups"])
            acc = {g: float(np.mean([r["groups"][g] for r in rows])) for g in names}
            counts = rows[0].get("counts", {})
            result["groups"][arm] = dict(accuracy=acc, counts=counts, worst=min(acc, key=acc.get))
    for arm in arms:
        picks = [c["arms"][arm]["selected"] for c in cells if "selected" in c["arms"][arm] and "_sel_" in arm]
        if picks:
            result["selection"][arm] = {k: picks.count(k) for k in sorted(set(picks))}
    sds = []
    for s in planned:
        v = [c["arms"][REFERENCE]["wga"] for c in cells if c["source"] == s and REFERENCE in c["arms"]]
        if len(v) > 1:
            sds.append(float(np.std(v, ddof=1)))
    result["reference_within_backbone_sd"] = float(np.mean(sds)) if sds else float("nan")
    result["erm_wga"] = erm_reference(sources)
    return result


def render(results, script_sha):
    lines = ["# Cross-dataset test of DPE post-hoc procedures", "",
             f"Analysis script SHA-256: `{script_sha}`. Official test labels. Differences in percentage points (pp) "
             "against full-validation DPE retrained with the same seeds; mean over split seeds within a backbone, then "
             "mean and pointwise 95% t interval over backbones. Decision: supported if mean > 0 and the interval excludes 0.", ""]
    for r in results:
        lines += [f"## {r['label']}", "",
                  f"Backbones: {len(r['backbones'])}; cells: {r['cells']}/{len(r['backbones']) * len(EXPECTED_SPLITS)}.",
                  ("**PARTIAL RUN: missing cells " + str(r["missing"]) + ". Do not report these numbers as the declared experiment.**")
                  if r["missing"] else "All declared cells complete.", ""]
        rows = [[c["id"] + (" (primary)" if c["primary"] else ""), f"`{c['arm']}` - `{c['control']}`", fmt(c["mean"], 2, True),
                 f"[{fmt(c['lo'], 2, True)}, {fmt(c['hi'], 2, True)}]", f"{c['positive']}/{c['n']}", c["decision"],
                 ", ".join(fmt(v, 2, True) for v in c["per_source"].values())] for c in r["contrasts"]]
        lines += [table(["contrast", "arms", "mean", "95% interval", "backbones > 0", "decision", "per backbone"], rows), ""]
        ref = r["arms"].get(REFERENCE, {})
        erm = list(r["erm_wga"].values())
        lines += [f"Full-validation DPE: WGA {fmt(ref.get('wga'))}%, overall {fmt(ref.get('overall'))}%; "
                  f"within-backbone SD over split seeds {fmt(r['reference_within_backbone_sd'])} pp; "
                  f"ERM backbone WGA {fmt(float(np.mean(erm))) if erm else 'n/a'}%.", ""]
        if r["groups"]:
            names = sorted(next(iter(r["groups"].values()))["accuracy"])
            counts = next(iter(r["groups"].values()))["counts"]
            rows = [[f"`{arm}`", *[fmt(v["accuracy"][g]) for g in names], v["worst"]] for arm, v in r["groups"].items()]
            lines += ["Per-group test accuracy (%), mean over backbones and split seeds:", "",
                      table(["arm", *[f"{g} (n={counts.get(g, '?')})" for g in names], "worst group"], rows), ""]
        rows = [[f"`{a}`", fmt(v["wga"]), fmt(v["overall"]), fmt(v["delta"], 2, True),
                 f"[{fmt(v['lo'], 2, True)}, {fmt(v['hi'], 2, True)}]", f"{v['positive']}/{v['n']}"]
                for a, v in r["arms"].items()]
        lines += ["All arms (supplementary):", "",
                  table(["arm", "WGA", "overall", "vs full-val DPE", "95% interval", "backbones > 0"], rows), ""]
        if r["selection"]:
            rows = [[f"`{a}`", ", ".join(f"{k}: {v}" for k, v in s.items())] for a, s in r["selection"].items()]
            lines += ["Settings chosen on validation data (cells):", "", table(["arm", "picks"], rows), ""]
    return "\n".join(lines)


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("outs", nargs="+", type=Path)
    p.add_argument("--report", type=Path, required=True)
    a = p.parse_args()
    script_sha = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    results = [analyse(o) for o in a.outs]
    a.report.mkdir(parents=True, exist_ok=True)
    text = render(results, script_sha)
    (a.report / "REPORT.md").write_text(text, encoding="utf-8")
    (a.report / "summary.json").write_text(json.dumps(dict(script_sha256=script_sha, results=results), indent=1,
                                                      default=float), encoding="utf-8")
    with (a.report / "contrasts.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["dataset", "label", "contrast", "arm", "control", "mean", "lo", "hi", "positive", "n", "decision"])
        for r in results:
            for c in r["contrasts"]:
                w.writerow([r["dataset"], r["label"], c["id"], c["arm"], c["control"], c["mean"], c["lo"], c["hi"],
                            c["positive"], c["n"], c["decision"]])
    print(text)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## Backbones

In [ ]:
# Continue from an attached earlier output of this notebook (same dataset, same mode family).
if not OUT.exists():
    for protocol in sorted(glob.glob('/kaggle/input/**/protocol.json', recursive=True)):
        p = Path(protocol).parent
        if p.name == OUT.name:
            shutil.copytree(p, OUT)
            print('continuing from', p)
            break
roots = [r for r in glob.glob('/kaggle/input/*')]
subprocess.run([sys.executable, '-m', 'xdpe.nested', 'discover', '--roots', *roots, '--dataset', DATASET,
                '--out', str(OUT), *(['--val-subsample', str(VAL_SUBSAMPLE)] if VAL_SUBSAMPLE else [])], cwd=CODE, check=True)
print(json.dumps([s['id'] for s in json.loads((OUT / 'sources.json').read_text())['sources']]))

## Run

In [ ]:
sys.path.insert(0, str(CODE))
import summarize
SPLITS = summarize.EXPECTED_SPLITS   # the declared split seeds; one definition for the run and the analysis
args = [sys.executable, '-m', 'xdpe.nested', 'run', '--out', str(OUT), '--workers', str(WORKERS),
        '--time-limit-hours', str(TIME_LIMIT_HOURS)]
if MODE == 'run':
    args += ['--splits', *map(str, SPLITS)]
else:
    first = json.loads((OUT / 'sources.json').read_text())['sources'][0]['id']
    args += ['--splits', '0', '--only', first, '--smoke' if MODE == 'smoke' else '--no-test']
started = time.time()
rc = subprocess.run(args, cwd=CODE).returncode
print(f'exit code {rc} after {(time.time() - started) / 3600:.2f} h')
for e in sorted((OUT / 'errors').glob('*.txt')) if (OUT / 'errors').exists() else []:
    print('ERROR', e.name); print(e.read_text()[-3000:])
if MODE != 'run':
    for p in sorted((OUT / 'smoke').rglob('*.json')):
        info = json.loads(p.read_text())
        print(p.relative_to(OUT), f"{info['seconds'] / 60:.1f} CPU-min (one worker)", 'arms:', len(info['arms']))

## Summary (complete runs only)

In [ ]:
sources = json.loads((OUT / 'sources.json').read_text())['sources']
expected = len(sources) * len(SPLITS)
done = len(list((OUT / 'results').glob('*/split*.json'))) if (OUT / 'results').exists() else 0
if MODE == 'run' and done == expected and len(sources) >= 5:
    subprocess.run([sys.executable, 'summarize.py', str(OUT), '--report', str(WORK / f'report-{tag}')], cwd=CODE, check=True)
elif MODE == 'run':
    print(f'{done}/{expected} cells complete ({len(sources)} backbones). Not summarised: PLAN Sect. 8.5 forbids looking at '
          'partial results. Save this version, attach its output and run again to continue.')

In [ ]:
import zipfile
bundle = WORK / f'xdpe-{tag}-{MODE}.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and 'checkpoints' not in p.parts:
            z.write(p, p.relative_to(WORK))
    report = WORK / f'report-{tag}'
    for p in (report.rglob('*') if report.exists() else []):
        z.write(p, p.relative_to(WORK))
    for p in CODE.rglob('*.py'):
        z.write(p, p.relative_to(WORK))
print('bundle', bundle, round(bundle.stat().st_size / 1e6, 1), 'MB (checkpoints stay in the working directory for resuming)')